In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ---------- SETTINGS ----------
initial_capital = 100000
cash = initial_capital
stop_loss_pct = 0.10
max_positions = 5
lookback_period = "10y"

trade_log = []
open_positions = {}

# ---------- LOAD ALL NSE STOCKS ----------
def load_nse_tickers():
    df = pd.read_csv("data/EQUITY_L.csv")

    symbols = (
        df["SYMBOL"]
        .dropna()
        .astype(str)
        .str.strip()
        .unique()
        .tolist()
    )

    tickers = []

    for symbol in symbols:
        if "&" in symbol:
            continue
        tickers.append(symbol + ".NS")

    return tickers


tickers = load_nse_tickers()
print("Total Indian stocks loaded:", len(tickers))

# ---------- UT BOT ----------
def compute_utbot(df, atr_period=1, multiplier=1):

    df["tr"] = np.maximum(
        df["High"] - df["Low"],
        np.maximum(
            abs(df["High"] - df["Close"].shift()),
            abs(df["Low"] - df["Close"].shift())
        )
    )

    df["atr"] = df["tr"].rolling(atr_period).mean()

    df["upper"] = df["Close"] - multiplier * df["atr"]
    df["lower"] = df["Close"] + multiplier * df["atr"]

    trend = [1]

    for i in range(1, len(df)):
        if df["Close"].iloc[i] > df["lower"].iloc[i-1]:
            trend.append(1)
        elif df["Close"].iloc[i] < df["upper"].iloc[i-1]:
            trend.append(-1)
        else:
            trend.append(trend[-1])

    df["trend"] = trend
    df["buy"] = (df["trend"] == 1) & (df["trend"].shift() == -1)
    df["sell"] = (df["trend"] == -1) & (df["trend"].shift() == 1)

    return df


# ---------- PRELOAD DATA ----------
all_data = {}

for ticker in tickers:

    print("Loading:", ticker)

    try:
        stock = yf.Ticker(ticker)
        df = stock.history(period=lookback_period)

        if df.empty:
            continue

        df = compute_utbot(df)

        info = stock.info
        eps = info.get("trailingEps")

        if eps is None:
            continue

        growth = info.get("earningsQuarterlyGrowth")
        if growth is None:
            growth = 0.05

        g = min(growth * 100, 12)

        intrinsic = eps * (8.5 + 2 * g)

        df["Intrinsic"] = intrinsic

        all_data[ticker] = df

    except:
        continue


# ---------- MASTER DATE INDEX ----------
all_dates = sorted(
    set(date for df in all_data.values() for date in df.index)
)

equity_curve = []

# ---------- PORTFOLIO BACKTEST ----------
for current_date in all_dates:

    # ---------- SELL FIRST ----------
    for ticker in list(open_positions.keys()):

        df = all_data[ticker]

        if current_date not in df.index:
            continue

        row = df.loc[current_date]
        pos = open_positions[ticker]

        stop_price = pos["entry_price"] * (1 - stop_loss_pct)

        if row["Close"] <= stop_price:
            exit_reason = "Stop Loss"

        elif row["sell"]:
            exit_reason = "UT Sell"

        else:
            continue

        exit_price = row["Close"]
        proceeds = pos["shares"] * exit_price
        profit = proceeds - pos["invested"]

        cash += proceeds

        trade_log.append({
            "Stock": ticker,
            "Entry Date": pos["entry_date"],
            "Exit Date": current_date,
            "Entry": pos["entry_price"],
            "Exit": exit_price,
            "Invested": pos["invested"],
            "Profit": profit,
            "Return %": (exit_price / pos["entry_price"] - 1) * 100,
            "Exit Reason": exit_reason
        })

        del open_positions[ticker]

    # ---------- BUY NEW POSITIONS ----------
    available_slots = max_positions - len(open_positions)

    if available_slots > 0:

        candidates = []

        for ticker, df in all_data.items():

            if ticker in open_positions:
                continue

            if current_date not in df.index:
                continue

            row = df.loc[current_date]

            if row["buy"] and row["Close"] < row["Intrinsic"]:
                discount = (row["Intrinsic"] - row["Close"]) / row["Intrinsic"]
                candidates.append((ticker, discount))

        # Most undervalued first
        candidates.sort(key=lambda x: x[1], reverse=True)

        for ticker, discount in candidates[:available_slots]:

            df = all_data[ticker]
            row = df.loc[current_date]

            allocation = cash / (max_positions - len(open_positions))

            if allocation <= 0:
                break

            shares = allocation / row["Close"]

            cash -= allocation

            open_positions[ticker] = {
                "entry_date": current_date,
                "entry_price": row["Close"],
                "shares": shares,
                "invested": allocation
            }

    # ---------- DAILY EQUITY ----------
    portfolio_value = cash

    for ticker, pos in open_positions.items():
        df = all_data[ticker]

        if current_date in df.index:
            portfolio_value += pos["shares"] * df.loc[current_date]["Close"]

    equity_curve.append(portfolio_value)


# ---------- RESULTS ----------
trades = pd.DataFrame(trade_log)

print("\nTotal Trades:", len(trades))

if len(trades) > 0:
    win_rate = len(trades[trades["Profit"] > 0]) / len(trades)
    print("Win Rate:", win_rate)

final_capital = equity_curve[-1]

print("\nInitial Capital:", initial_capital)
print("Final Capital:", final_capital)
print("Total Return %:", (final_capital / initial_capital - 1) * 100)

# ---------- TRADE LOG ----------
if not trades.empty:

    trades = trades.sort_values("Exit Date")

    trades["Cumulative Profit"] = trades["Profit"].cumsum()
    trades["Equity"] = initial_capital + trades["Cumulative Profit"]

    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)

    print("\nFULL TRADE LOG:\n")
    print(trades)

    trades.to_csv("trade_log.csv", index=False)
    print("\nTrade log saved as trade_log.csv")

# ---------- EQUITY CURVE ----------
plt.figure()
plt.plot(equity_curve)
plt.title("Diversified Portfolio Equity Curve")
plt.xlabel("Time")
plt.ylabel("Portfolio Value")
plt.show()

# ---------- RISK / REWARD ----------
if not trades.empty:

    avg_win = trades[trades["Profit"] > 0]["Profit"].mean()
    avg_loss = trades[trades["Profit"] < 0]["Profit"].mean()

    print("\nAverage Win:", avg_win)
    print("Average Loss:", avg_loss)

    if avg_loss != 0:
        print("Risk Reward Ratio:", abs(avg_win / avg_loss))

    print("\nBest Trade:", trades["Profit"].max())
    print("Worst Trade:", trades["Profit"].min())

    print("\nExit Reason Breakdown:")
    print(trades["Exit Reason"].value_counts())

    print("\nAverage Return by Exit Type:")
    print(trades.groupby("Exit Reason")["Return %"].mean())